### Missing Values

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data = pd.read_csv("melbournehousingmarket/Melbourne_housing_FULL.csv")

In [ ]:
data = data[data["Price"].notna()]

In [ ]:
y = data.Price

In [ ]:
melb_predictors = data.drop(["Price"], axis = 1)

In [ ]:
X = melb_predictors.select_dtypes(exclude = ['object'])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size = 0.8, random_state = 0)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor()
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

#### 1) A Simple Option: Drop Columns with Missing Values

In [ ]:
cols_with_missing = [col for col in X_train.columns if X_train[col].isnull().any()]

In [ ]:
reduced_X_train = X_train.drop(cols_with_missing, axis = 1)
reduced_X_valid = X_valid.drop(cols_with_missing, axis = 1)

In [ ]:
print("MAE from Approach 1 (Dropping Columns with Missing Values):")
print(score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid))

#### 2) A Better Option: Imputation

In [ ]:
from sklearn.impute import SimpleImputer

my_imputer = SimpleImputer()
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_valid = pd.DataFrame(my_imputer.transform(X_valid))

imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns

print("MAE from Approach 2 (Imputation):")
print(score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid))

#### 3) An Extension To Imputation

In [ ]:
X_train_plus = X_train.copy()
X_valid_plus = X_valid.copy()

for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_valid_plus[col + '_was_missing'] = X_valid_plus[col].isnull()
    
my_imputer = SimpleImputer()
imputed_X_train_plus = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
imputed_X_valid_plus = pd.DataFrame(my_imputer.transform(X_valid_plus))

imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_valid_plus.columns = X_valid_plus.columns

print("MAE from Approach 3 (An Extension to Imputation):")
print(score_dataset(imputed_X_train_plus, imputed_X_valid_plus, y_train, y_valid))

### Categorical Variables

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv("melbournehousingmarket/Melbourne_housing_FULL.csv")
data = data.dropna()

y = data.Price

X = data.drop(["Price"], axis = 1)

X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, train_size = 0.8, random_state = 0)

cols_with_missing = [col for cols in X_train_full.columns if X_train_full[col].isnull().any()]

X_train_full.drop(cols_with_missing, axis = 1, inplace = True)
X_valid_full.drop(cols_with_missing, axis = 1, inplace = True)

low_cardinality_cols = [cname for cname in X_train_full.columns if (X_train_full[cname].nunique() < 10) and (X_train_full[cname].dtype == "object")]

numerical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].dtype in ['int64', 'float64']]

my_cols = low_cardinality_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()

In [ ]:
X_train.head()

In [ ]:
s = (X_train.dtypes == 'object')
object_cols = list(s[s].index)
print("Categorical variables:")
print(object_cols)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor()
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    mae = mean_absolute_error(y_valid, preds)
    return mae

#### 1) Drop Categorical Variables

In [ ]:
drop_X_train = X_train.select_dtypes(exclude = "object")
drop_X_valid = X_valid.select_dtypes(exclude = "object")

print("MAE from Approach 1 (Drop Categorical Variables):")
print(score_dataset(drop_X_train, drop_X_valid, y_train, y_valid))

#### 2) Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_X_train = X_train.copy()
label_X_valid = X_valid.copy()

label_encoder = LabelEncoder()
for col in object_cols:
    label_X_train[col] = label_encoder.fit_transform(X_train[col])
    label_X_valid[col] = label_encoder.transform(X_valid[col])
    
print("MAE from Approach 2 (Label Encoding):")
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

#### 3) One-Hot Encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder

OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse = False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[object_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[object_cols]))

OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

num_X_train = X_train.drop(object_cols, axis = 1)
num_X_valid = X_valid.drop(object_cols, axis = 1)

OH_X_train = pd.concat([num_X_train, OH_cols_train], axis = 1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis = 1)

print("MAE from Approach 3 (One-Hot Encoding):") 
print(score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))

### Pipeline

Pipelines are a simple way to keep your data preprocessing and modeling code organized. Specifically, a pipeline bundles preprocessing and modeling steps so you can use the whole bundle as if it were a single step.

Many data scientists hack together models without pipelines, but pipelines have some important benefits. Those include:

**Cleaner Code**: Accounting for data at each step of preprocessing can get messy. With a pipeline, you won't need to manually keep track of your training and validation data at each step.

**Fewer Bugs**: There are fewer opportunities to misapply a step or forget a preprocessing step.

**Easier to Productionize**: It can be surprisingly hard to transition a model from a prototype to something deployable at scale. We won't go into the many related concerns here, but pipelines can help.

**More Options for Model Validation**: You will see an example in the next tutorial, which covers cross-validation.

In [ ]:
import numpy as np
import pandas as pd

data = pd.read_csv("melbournehousingmarket/Melbourne_housing_FULL.csv")
data.dropna(inplace = True)

y = data["Price"]
X = data.drop(columns = ["Price"])

from sklearn.model_selection import train_test_split
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, train_size = 0.8, random_state = 1)

categorical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].dtype == "object" and X_train_full[cname].nunique() < 10]

numerical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].dtype in ["int64", "float64"]]

my_cols = categorical_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()

In [ ]:
#### Step 1: Define Preprocessing Steps

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numerical_transformer = SimpleImputer(strategy = 'constant')

categorical_transformer = Pipeline(steps = [('imputer', SimpleImputer(strategy = 'most_frequent')), ('onehot', OneHotEncoder(handle_unknown = 'ignore'))])

preprocessor = ColumnTransformer(transformers = [('num', numerical_transformer, numerical_cols), ('cat', categorical_transformer, categorical_cols)])

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators = 100, random_state = 1)

In [ ]:
from sklearn.metrics import mean_absolute_error

my_pipeline = Pipeline(steps = [('preprocessor', preprocessor), ('model', model)])

my_pipeline.fit(X_train, y_train)

preds = my_pipeline.predict(X_valid)

score = mean_absolute_error(y_valid, preds)
print("MAE:", score)

In [ ]:
import numpy as np
import pandas as pd

data = pd.read_csv("melbournehousingmarket/Melbourne_housing_FULL.csv")
data = data[data["Price"].notna()]

y = data["Price"]
cols_to_use = ['Rooms', 'Distance', 'Landsize', 'BuildingArea', 'YearBuilt']

X = data[cols_to_use]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

my_pipeline = Pipeline(steps = [("preprocessor", SimpleImputer()), ("model", RandomForestRegressor(n_estimators = 50, random_state = 1))])

In [ ]:
from sklearn.model_selection import cross_val_score

scores = -1 * cross_val_score(my_pipeline, X, y, cv = 5, scoring = "neg_mean_absolute_error")
print("MAE Scores: ", scores)

In [ ]:
print("Average MAE score (across experiments):")
print(scores.mean())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Read the data
data = pd.read_csv("melbournehousingmarket/Melbourne_housing_FULL.csv")
data = data[data["Price"].notna()]

# Select subset of predictors
cols_to_use = ['Rooms', 'Distance', 'Landsize', 'BuildingArea', 'YearBuilt']
X = data[cols_to_use]

# Select target
y = data.Price

# Separate data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(X, y)

In [ ]:
from xgboost import XGBRegressor

my_model = XGBRegressor()
my_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error

predictions = my_model.predict(X_valid)
print("Mean Absolute Error: " + str(mean_absolute_error(predictions, y_valid)))